# Assistente Tecnico Databricks - RAG Conectado a Documentacao Oficial

Este projeto implementa um assistente inteligente especializado no ecossistema Databricks e Engenharia de Dados, com ingestao direta de paginas do portal oficial de documentacao: https://docs.databricks.com/aws/en/

O sistema utiliza busca semantica densa com embeddings oficiais do Google (`gemini-embedding-001`), banco vetorial local persistente (`ChromaDB`) e sintese contextualizada com o `gemini-3.5-flash-lite`, citando os links oficiais de onde cada resposta foi extraida.

### Componentes da Solucao:
- **Ingestao Web Oficial:** Parser nativo que extrai artigos diretamente de `docs.databricks.com`.
- **Modelo de Embeddings:** Google `gemini-embedding-001` via SDK oficial `google-genai`.
- **Banco Vetorial:** `ChromaDB` em modo persistente local com metadados de URL e secao.
- **Modelo de Linguagem:** Google `gemini-3.5-flash-lite` com regras anti-alucinacao e retry automatico.
- **Interface:** `Gradio` com chat e suporte para indexar novos links da Databricks sob demanda.

## 1. Instalacao das Dependencias
Instalamos o SDK do Google GenAI, o ChromaDB e o Gradio.

In [ ]:
%%capture
# Instalacao dos pacotes sem avisos de conflito do Colab
!pip install -q google-genai chromadb gradio

## 2. Autenticacao com a API do Google Gemini
Recuperacao segura da chave via Colab Secrets (`userdata.get`) ou variavel de ambiente.

In [ ]:
import os
import json
from google import genai

def extrair_api_key(chave_raw):
    """Extrai e sanitiza a GEMINI_API_KEY se fornecida como str, dict, json ou outro formato."""
    if not chave_raw:
        return None
    if isinstance(chave_raw, dict):
        for campo in ["GEMINI_API_KEY", "api_key", "key", "token", "response", "value"]:
            if campo in chave_raw and isinstance(chave_raw[campo], str):
                return chave_raw[campo].strip()
        for v in chave_raw.values():
            if isinstance(v, str) and len(v.strip()) > 5:
                return v.strip()
            elif isinstance(v, dict):
                res = extrair_api_key(v)
                if res:
                    return res
        return None
    if isinstance(chave_raw, str):
        chave_raw = chave_raw.strip()
        if (chave_raw.startswith("{") and chave_raw.endswith("}")) or (chave_raw.startswith("[") and chave_raw.endswith("]")):
            try:
                dados = json.loads(chave_raw)
                return extrair_api_key(dados)
            except Exception:
                pass
        return chave_raw
    return str(chave_raw).strip() if chave_raw else None

# 1. Recupera dos Secrets do Colab ou do ambiente
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("GEMINI_API_KEY")

api_key = extrair_api_key(api_key)

# 2. Se nao encontrou, solicita via getpass ou input com protecao total
if not api_key:
    try:
        import getpass
        bruto = getpass.getpass("Digite sua GEMINI_API_KEY: ")
        api_key = extrair_api_key(bruto)
    except Exception:
        pass

if not api_key:
    try:
        bruto = input("Digite sua GEMINI_API_KEY: ")
        api_key = extrair_api_key(bruto)
    except Exception:
        pass

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY nao configurada! "
        "No Google Colab, clique no icone da chave (Secrets) no menu lateral esquerdo, "
        "adicione uma chave com o nome 'GEMINI_API_KEY' e ATIVE a opcao 'Notebook access'."
    )

client = genai.Client(api_key=api_key)
print("Cliente Gemini configurado com sucesso.")

## 3. Ingestao Automatica Direta via Sitemap Oficial (docs.databricks.com/sitemap.xml)
O notebook conecta diretamente ao sitemap oficial da Databricks, varre milhares de URLs disponiveis,
filtra automaticamente os topicos de Engenharia de Dados (Lakehouse, Delta, Warehouses, Unity Catalog, Ingestao, DLT) e extrai o conteudo limpo em tempo real.

In [ ]:
import os
import re
import urllib.request
from html.parser import HTMLParser

class DatabricksDocParser(HTMLParser):
    """Parser HTML para extrair conteudo limpo das paginas da Databricks."""
    def __init__(self):
        super().__init__()
        self.in_article = False
        self.in_script = False
        self.text_parts = []
        self.title = ""
        self.in_title = False

    def handle_starttag(self, tag, attrs):
        if tag in ["script", "style", "nav", "footer", "header", "noscript"]:
            self.in_script = True
        if tag in ["article", "main"]:
            self.in_article = True
        if tag == "title":
            self.in_title = True
        if tag in ["h1", "h2", "h3", "p", "li"]:
            self.text_parts.append("\n")

    def handle_endtag(self, tag):
        if tag in ["script", "style", "nav", "footer", "header", "noscript"]:
            self.in_script = False
        if tag in ["article", "main"]:
            self.in_article = False
        if tag == "title":
            self.in_title = False

    def handle_data(self, data):
        if self.in_title:
            self.title += data.strip()
        if self.in_article and not self.in_script:
            self.text_parts.append(data)

def extrair_pagina_databricks(url):
    """Baixa uma URL de docs.databricks.com e extrai o texto do artigo."""
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=15) as resp:
            html = resp.read().decode("utf-8")
        
        parser = DatabricksDocParser()
        parser.feed(html)
        texto_bruto = "".join(parser.text_parts)
        linhas = [linha.strip() for linha in texto_bruto.split("\n") if linha.strip()]
        texto_limpo = "\n".join(linhas)
        
        titulo = parser.title.replace(" | Databricks on AWS", "").replace(" | Databricks", "").strip()
        return {"url": url, "titulo": titulo or url, "conteudo": texto_limpo}
    except Exception as e:
        return None

def buscar_urls_sitemap(max_paginas=30):
    """Varre docs.databricks.com/sitemap.xml e seleciona paginas de Engenharia de Dados."""
    sitemap_url = "https://docs.databricks.com/sitemap.xml"
    print(f"Lendo sitemap oficial: {sitemap_url}...")
    headers = {"User-Agent": "Mozilla/5.0"}
    req = urllib.request.Request(sitemap_url, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as resp:
        xml = resp.read().decode("utf-8")
    
    todas = re.findall(r"<loc>(https://docs\.databricks\.com/aws/en/[^<]+)</loc>", xml)
    print(f"Total de URLs no sitemap: {len(todas)}")
    
    topicos = ["/lakehouse/", "/delta/", "/data-governance/", "/ingestion/", "/compute/sql-warehouse/", "/delta-live-tables/", "/optimizations/"]
    excluidos = ["/archive/", "/release-notes/", "/error-messages/"]
    
    selecionadas = []
    for u in todas:
        if any(exc in u for exc in excluidos):
            continue
        if any(top in u for top in topicos):
            selecionadas.append(u)
            
    print(f"URLs filtradas para Engenharia de Dados: {len(selecionadas)}")
    return selecionadas[:max_paginas]

# Numero de paginas do sitemap para carregar nesta execucao
MAX_PAGINAS_SITEMAP = 35
urls_sitemap = buscar_urls_sitemap(max_paginas=MAX_PAGINAS_SITEMAP)

print(f"\nExtraindo conteudo de {len(urls_sitemap)} paginas oficiais da Databricks...")
documentos_carregados = []
for i, url in enumerate(urls_sitemap, start=1):
    doc = extrair_pagina_databricks(url)
    if doc and len(doc["conteudo"]) > 200:
        documentos_carregados.append(doc)
        if i % 5 == 0 or i == len(urls_sitemap):
            print(f"Progresso: {i}/{len(urls_sitemap)} paginas extraidas ({doc['titulo'][:45]}...)")

# Carrega tambem os modulos conceituais locais
DOCS_DIR = "data/databricks_docs"
if os.path.exists(DOCS_DIR):
    for f in os.listdir(DOCS_DIR):
        if f.endswith(".md"):
            with open(os.path.join(DOCS_DIR, f), "r", encoding="utf-8") as arq:
                titulo_doc = f.replace(".md", "").replace("_", " ").title()
                documentos_carregados.append({
                    "url": f"https://docs.databricks.com/aws/en/ (Doc: {f})",
                    "titulo": f"Databricks Concept - {titulo_doc}",
                    "conteudo": arq.read()
                })

print(f"\nTotal de fontes tecnicas carregadas: {len(documentos_carregados)}")

## 4. Chunking Semantico com Metadados
Dividimos os documentos por secoes tecnicas (titulos em Markdown `##`), preservando o contexto e associando metadados como nome do arquivo e titulo da secao em cada bloco.

In [ ]:
def dividir_em_chunks_web(documentos, tamanho=900, overlap=120):
    """
    Divide os textos das paginas web em blocos com metadados de URL e titulo.
    """
    chunks = []
    passo = tamanho - overlap
    
    for doc in documentos:
        texto = doc["conteudo"]
        url = doc["url"]
        titulo = doc["titulo"]
        
        for inicio in range(0, max(1, len(texto)), passo):
            trecho = texto[inicio:inicio + tamanho].strip()
            if len(trecho) > 60:
                chunks.append({
                    "titulo": titulo,
                    "url": url,
                    "texto": trecho
                })
                
    return chunks

chunks = dividir_em_chunks_web(documentos_carregados)
print(f"Total de chunks gerados da documentacao oficial: {len(chunks)}")
if chunks:
    print(f"Exemplo:\nTitulo: {chunks[0]['titulo']}\nURL: {chunks[0]['url']}\nTexto: {chunks[0]['texto'][:180]}...")

## 5. Embeddings e Banco Vetorial com ChromaDB
Utilizamos o modelo Google `gemini-embedding-001` para converter cada trecho de texto em um vetor semantico de 768 dimensoes.
Os vetores e metadados sao gravados no `ChromaDB` em modo persistente (`./chroma_db`), garantindo que novas consultas nao precisem recalcular os embeddings.

In [ ]:
import chromadb
import time

# Inicializa o cliente do ChromaDB com armazenamento local
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(
    name="databricks_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

def gerar_embedding(texto, max_tentativas=3):
    """Gera embedding individual garantindo exatamente 1 vetor por chamada."""
    for mod in ["gemini-embedding-001", "gemini-embedding-2"]:
        for tentativa in range(max_tentativas):
            try:
                resp = client.models.embed_content(model=mod, contents=texto)
                if resp.embeddings and len(resp.embeddings) > 0:
                    return resp.embeddings[0].values
            except Exception:
                time.sleep(1.5 * (tentativa + 1))
    raise RuntimeError("Falha ao gerar embedding.")

def obter_embedding(texto):
    return gerar_embedding(texto)

# Indexa os chunks caso a colecao ainda nao esteja totalmente populada
total_existente = collection.count()
if total_existente < len(chunks):
    print(f"Banco possui {total_existente} chunks. Retomando indexacao ate {len(chunks)} chunks...")
    
    tamanho_lote = 10
    for i in range(total_existente, len(chunks), tamanho_lote):
        lote = chunks[i:i + tamanho_lote]
        
        # Gera 1 vetor por texto garantindo 100% de paridade matematica
        vetores = []
        for c in lote:
            txt_completo = f"Titulo: {c['titulo']} | URL: {c['url']}\n{c['texto']}"
            vetores.append(gerar_embedding(txt_completo))
            time.sleep(0.04)
            
        ids = [f"chunk_{i + j}" for j in range(len(lote))]
        documents = [c["texto"] for c in lote]
        metadatas = [{"titulo": c["titulo"], "url": c["url"]} for c in lote]
        
        collection.upsert(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=vetores
        )
        print(f"Progresso: {min(i + len(lote), len(chunks))}/{len(chunks)} chunks gravados com sucesso.")
        time.sleep(0.1)
        
    print(f"\nIndexacao concluida com sucesso! Total no banco: {collection.count()} registros.")
else:
    print(f"Banco vetorial ja indexado ({total_existente} chunks disponiveis no ChromaDB).")

## 6. Funcao de Busca Semantica no ChromaDB
Calcula a proximidade angular (cosseno) entre a intencao da pergunta e os trechos indexados.

In [ ]:
def buscar_documentos(pergunta, top_k=4):
    """
    Busca semantica no ChromaDB com acesso seguro aos metadados.
    """
    vetor_pergunta = obter_embedding(pergunta)
    
    res = collection.query(
        query_embeddings=[vetor_pergunta],
        n_results=top_k
    )
    
    resultados = []
    if res["documents"] and len(res["documents"][0]) > 0:
        for i in range(len(res["documents"][0])):
            meta = res["metadatas"][0][i] if res.get("metadatas") else {}
            titulo = meta.get("titulo") or meta.get("documento") or "Documentacao Databricks"
            url = meta.get("url") or meta.get("secao") or "https://docs.databricks.com/aws/en/"
            dist = res["distances"][0][i] if "distances" in res else None
            
            resultados.append({
                "texto": res["documents"][0][i],
                "titulo": titulo,
                "url": url,
                "distancia": dist
            })
            
    return resultados

print("Funcao de busca semantica configurada com acesso seguro aos metadados.")

## 7. Pipeline RAG com Fallback Inteligente (Grounding Oficial)
Integramos a recuperacao semantica local no ChromaDB com sintese do Gemini. Se a informacao nao for encontrada na base local pre-indexada, o assistente aciona automaticamente o **Google Search Grounding** restrito a `docs.databricks.com` para varrer a documentacao oficial ao vivo sem custo no tier gratuito.

In [ ]:
import time
from google.genai import types

MODELOS_LLM = ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-3.5-flash-lite"]

def pesquisar_documentacao_ao_vivo(pergunta, max_tentativas=3):
    """
    Fallback inteligente: Consulta a documentacao oficial do Databricks ao vivo
    via Google Search Grounding nativo do Gemini (sem custo no tier gratuito).
    """
    prompt_web = f"""
Voce e um especialista em Engenharia de Dados focado no ecossistema Databricks.
Pesquise exclusivamente na documentacao oficial da Databricks (site:docs.databricks.com) para responder a pergunta tecnica do usuario.

Diretrizes:
1. Responda em portugues de forma estruturada, didatica e completa.
2. Cite explicitamente as URLs e paginas oficiais consultadas em docs.databricks.com.
3. Se a pergunta for totalmente fora do escopo do Databricks ou engenharia de dados (ex: futebol, culinaria, fofocas), responda estritamente:
   "Nao encontrei essa informacao na documentacao oficial da Databricks."
4. Se o tema envolver configuracao, comandos ou codigo no Databricks, inclua exemplos praticos.

PERGUNTA:
{pergunta}
"""
    config_grounding = types.GenerateContentConfig(
        tools=[{"google_search": {}}],
        temperature=0.2,
    )
    
    for modelo in MODELOS_LLM:
        for tentativa in range(max_tentativas):
            try:
                resposta = client.models.generate_content(
                    model=modelo,
                    contents=prompt_web,
                    config=config_grounding
                )
                texto = resposta.text.strip()
                if texto:
                    if "nao encontrei essa informacao na documentacao oficial" in texto.lower():
                        return texto
                    return f"🌐 **[Informação recuperada ao vivo em docs.databricks.com]**\n\n{texto}"
            except Exception as e:
                erro_str = str(e)
                if "503" in erro_str or "429" in erro_str or "high demand" in erro_str.lower():
                    time.sleep(2 * (tentativa + 1))
                    continue
                break
                
    return "Nao foi possivel consultar a documentacao online no momento. Tente novamente em instantes."

def perguntar_databricks(pergunta, top_k=4, max_tentativas=3, threshold_distancia=0.68):
    """
    Ciclo RAG Hibrido:
    1. Busca local nos documentos previamente indexados no ChromaDB.
    2. Se nao encontrar ou se a similaridade for baixa, aciona a varredura ao vivo na documentacao oficial.
    """
    trechos = buscar_documentos(pergunta, top_k=top_k)
    
    # Avalia se a busca local tem boa relevancia semantica
    encontrou_local = False
    if trechos:
        menor_distancia = trechos[0].get("distancia")
        # Se a menor distancia for menor que o threshold, temos trecho relevante
        if menor_distancia is None or menor_distancia < threshold_distancia:
            encontrou_local = True
            
    if not encontrou_local:
        print("-> Assunto fora da base local. Acionando varredura ao vivo em docs.databricks.com...")
        return pesquisar_documentacao_ao_vivo(pergunta)
        
    contexto = "\n\n".join([
        f"[Fonte Oficial Local: {t['titulo']} | Link: {t['url']}]\n{t['texto']}"
        for t in trechos
    ])
    
    prompt = f"""
Voce e um especialista em Engenharia de Dados focado no ecossistema Databricks.
Responda a pergunta do usuario utilizando as informacoes tecnicas do CONTEXTO abaixo.

Diretrizes de resposta:
1. Baseie-se nas informacoes presentes no CONTEXTO.
2. Se a pergunta for sobre termos amplos (como 'warehouse', 'data warehouse' ou 'SQL warehouse'), esclareca o conceito e como ele se aplica no Databricks.
3. Cite explicitamente a fonte oficial ou link utilizado na resposta.
4. Responda em portugues de forma estruturada, didatica e completa.
5. Se o CONTEXTO nao responder adequadamente a pergunta ou for insuficiente, responda exatamente:
   "BUSCAR_DOCS_ONLINE"

CONTEXTO:
{contexto}

PERGUNTA:
{pergunta}
"""
    
    for modelo in MODELOS_LLM:
        for tentativa in range(max_tentativas):
            try:
                resposta = client.models.generate_content(
                    model=modelo,
                    contents=prompt
                )
                texto_resposta = resposta.text.strip()
                
                # Se o proprio modelo notar que o contexto local nao basta, aciona fallback
                if "BUSCAR_DOCS_ONLINE" in texto_resposta or "nao encontrei essa informacao" in texto_resposta.lower():
                    print("-> Contexto local insuficiente. Acionando varredura ao vivo em docs.databricks.com...")
                    return pesquisar_documentacao_ao_vivo(pergunta)
                    
                return f"📚 **[Base de Conhecimento Local]**\n\n{texto_resposta}"
            except Exception as e:
                erro_str = str(e)
                if "503" in erro_str or "429" in erro_str or "high demand" in erro_str.lower():
                    time.sleep(2 * (tentativa + 1))
                    continue
                break
                
    return "Servidores da Google temporariamente ocupados. Por favor, tente novamente em alguns segundos."

print("Pipeline RAG com Fallback Inteligente (ChromaDB + Docs Databricks ao vivo) configurado.")


## 8. Bateria de Testes de Engenharia de Dados
Validacao com perguntas reais sobre arquitetura, governanca, otimizacao e um teste anti-alucinacao.

In [ ]:
import time

testes = [
    {
        "titulo": "Pergunta 1 - Arquitetura Medalhao (Base Local)",
        "pergunta": "Qual e a responsabilidade de cada camada (Bronze, Silver e Gold) na arquitetura Medalhao do Databricks?"
    },
    {
        "titulo": "Pergunta 2 - Funcionamento Interno do Delta Lake (Base Local)",
        "pergunta": "Como o log de transacoes (_delta_log) e os checkpoints garantem propriedades ACID no Delta Lake?"
    },
    {
        "titulo": "Pergunta 3 - Governanca com Unity Catalog (Base Local)",
        "pergunta": "Como funciona o namespace de tres niveis no Unity Catalog e quais as vantagens sobre o Hive Metastore?"
    },
    {
        "titulo": "Pergunta 4 - Varredura ao Vivo (Fallback Grounding em docs.databricks.com)",
        "pergunta": "Como funciona o recurso de Liquid Clustering no Databricks e como habilita-lo no Delta Lake?"
    },
    {
        "titulo": "Pergunta 5 - Teste Anti-Alucinacao (Fora do Escopo)",
        "pergunta": "Quem foi o campeao da Copa Libertadores da America em 2019?"
    }
]

print("=" * 70)
print("EXECUCAO DOS TESTES DO ASSISTENTE DATABRICKS")
print("=" * 70)

for t in testes:
    print(f"\n[{t['titulo']}]")
    print(f"Pergunta: {t['pergunta']}")
    resp = perguntar_databricks(t["pergunta"])
    print(f"\nResposta:\n{resp}")
    print("-" * 70)
    time.sleep(2)  # Pausa para respeitar os limites de requisicao da API gratuita


## 9. Interface Interativa com Gradio
Interface web para tirar duvidas em tempo real sobre Databricks e Engenharia de Dados.

In [ ]:
import gradio as gr

def responder_chat(mensagem, historico):
    if not mensagem.strip():
        return "Por favor, digite uma pergunta valida."
    return perguntar_databricks(mensagem)

demo = gr.ChatInterface(
    fn=responder_chat,
    title="Assistente Tecnico Databricks - RAG Especialista",
    description=(
        "Tire duvidas tecnicas sobre arquitetura Medalhao, Delta Lake, Unity Catalog, "
        "Delta Live Tables e otimizacao com Photon. "
        "Respostas geradas com base estrita na documentacao oficial com citacao de fontes."
    ),
    examples=[
        "Qual a diferenca entre a camada Bronze e Silver?",
        "Como o Delta Lake lida com transacoes concorrentes?",
        "O que e o namespace de tres niveis no Unity Catalog?",
        "Quais as vantagens do motor Photon sobre a JVM tradicional?"
    ]
)

demo.launch(share=True, debug=False)